## Mapping Text and Image Embeddings

In [ ]:
# Importing useful dependencies
import io
import torch
import boto3
import pickle
import random
import chromadb
import open_clip
import numpy as np
from PIL import Image
from io import BytesIO
from sklearn.linear_model import Ridge
from sklearn.preprocessing import normalize

# Set a seed for reproducibility
def set_seed(SEED=42):
  random.seed(SEED)
  np.random.seed(SEED)
  torch.manual_seed(SEED)
  torch.cuda.manual_seed_all(SEED)

set_seed(10721)

In [ ]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url="http://127.0.0.1:9000", # MinIO API endpoint
    aws_access_key_id="minioadmin", # User name
    aws_secret_access_key="minioadmin", # Password
)

In [ ]:
# Connect to the server (Docker Container)
client = chromadb.HttpClient(host="localhost", port=8000)

# Get the texts and images collections
collection_texts = client.create_collection(name="texts", get_or_create=True, embedding_function=None)
collection_images = client.create_collection(name="images", get_or_create=True, embedding_function=None)

# Create or get the collection named "texts_images" to store embeddings of images and texts
collection_texts_images = client.create_collection(name="texts_images", get_or_create=True, embedding_function=None)

# Create or get auxiliary collections for testing
collection_texts_another = client.create_collection(name="texts_another", get_or_create=True, embedding_function=None)
collection_texts_images_another = client.create_collection(name="texts_images_another", get_or_create=True, embedding_function=None)

Based on the professor's feedback, we experimented with using two separate models to generate embeddings for images and texts. The image embeddings had a dimensionality of 768, while the text embeddings were 512-dimensional. To align these representations, we trained a linear projector using Ridge regression to map the 512-dimensional text embeddings into the 768-dimensional image embedding space.

However, when we performed a similarity search using the query "NieR:Automata", the results were unsatisfactory, the actual game cover did not appear among the top 10 retrieved images. Consequently, we decided to revert to using a single model for generating embeddings for both texts and images. This time, we opted for a larger CLIP model (approximately two to three times bigger than the one we used in the first part of the project) to improve embedding quality and cross-modal alignment.

### A Discarded Approach

In [ ]:
# Function that returns all embeddings in a collection
def retrieve_embeddings(collection):
    embeddings = []
    results = collection.get(include=["documents", "embeddings"])
    for i in range(len(results["documents"])):
        embeddings.append(results["embeddings"][i])
    return embeddings

In [ ]:
# Retrieve text embeddings
text_embeddings = retrieve_embeddings(collection_texts_another) # Embeddings of size 512
# Retrieve image embeddings
image_embeddings = retrieve_embeddings(collection_images) # Embeddings of size 768

In [ ]:
set_seed(10721)

# Determine the number of pairs for training
sample_size = min(len(text_embeddings), len(image_embeddings))

# Normalize embeddings
Xn = normalize(text_embeddings[0:sample_size], axis=1)
Yn = normalize(image_embeddings[0:sample_size], axis=1)

# Train linear projector
ridge = Ridge(alpha=1.0, fit_intercept=True)
ridge.fit(Xn, Yn)

# Save the trained projector
with open("textdim_to_imagedim.pkl", "wb") as f:
    pickle.dump(ridge, f)

In [ ]:
# Check the coefficients of the linear projector
print("Ridge coef shape:", ridge.coef_.shape) # (D_img, D_txt)

# Project the text embeddings to the space of image embeddings
x = Xn # All normalized text embeddings
y_pred = ridge.predict(x) # shape (1, D_img)
print("y_pred shape:", y_pred.shape)

In [ ]:
# 1. Fetch all embeddings from the "images" collection
all_images = collection_images.get(include=["embeddings", "documents"])

# 2. Insert them into the new "texts_images_another" collection
collection_texts_images_another.add(
    ids=all_images["ids"],
    embeddings=all_images["embeddings"],
    documents=all_images["documents"],
    metadatas=[{"type": "image"} for _ in range(len(all_images["ids"]))]
)

In [ ]:
# 3. Fetch all embeddings from the "texts_another" collection
all_texts = collection_texts_another.get(include=["embeddings", "documents"])

# 4. Insert the projected text embeddings into the new "texts_images_another" collection
collection_texts_images_another.add(
    ids=all_texts["ids"],
    embeddings=y_pred,
    documents=all_texts["documents"],
    metadatas=[{"type": "text"} for _ in range(len(all_texts["ids"]))]
)

In [ ]:
# Just in case our device has gpu
device = "cuda" if torch.cuda.is_available() else "cpu"

# In case we want to use a different model for generating text embeddings
model_another, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained='openai')
tokenizer_another = open_clip.get_tokenizer("ViT-B-32")
model_another.to(device)

In [ ]:
@torch.no_grad()
# The next function returns the embedding of the given text
def embed_text(model, tokenizer, texts: str):
    tokens = tokenizer([texts]).to(device) # tokenized batch
    feats = model.encode_text(tokens)
    feats = feats / feats.norm(dim=-1, keepdim=True) # normalize
    return feats.cpu().numpy()[0]
# We can use this function to retrieve an image from our bucket in PIL Image format
def get_image(bucket, key):
    resp = s3.get_object(Bucket=bucket, Key=key)
    body = resp["Body"].read()
    img = Image.open(io.BytesIO(body))
    return img
# We can use this function to retrieve an text from our bucket
def get_text(bucket, key):
    resp = s3.get_object(Bucket=bucket, Key=key)
    body = resp["Body"].read()
    text = body.decode("utf-8")
    return text

# Similarity search by the textual query "NieR:Automata"
query_text = "Games similar to NieR:Automata"
q_vec = ridge.predict([embed_text(model_another, tokenizer_another, query_text)])[0]

res_texts = collection_texts_images_another.query(
    query_embeddings=[q_vec],
    n_results=1,
    where={"type": "text"}, # Filter by metadata type
    include=["documents", "distances"]
)

res_images = collection_texts_images_another.query(
    query_embeddings=[q_vec],
    n_results=10,
    where={"type": "image"}, # Filter by metadata type
    include=["documents", "distances"]
)

In [ ]:
# Diplay top 1 similar text
get_text("trusted-zone", res_texts['documents'][0][0].split("/", 1)[1])

In [ ]:
# Diplay top 10 similar images
for i in range(10):
    display(get_image("trusted-zone", res_images['documents'][0][i].split("/", 1)[1]))

### Put Text and Image Embeddings into the Same Collection

In [ ]:
# 1. Fetch all embeddings from the "images" collection
all_images = collection_images.get(include=["embeddings", "documents"])

# 2. Insert them into the new "texts_images" collection
collection_texts_images.add(
    ids=all_images["ids"],
    embeddings=all_images["embeddings"],
    documents=all_images["documents"],
    metadatas=[{"type": "image"} for _ in range(len(all_images["ids"]))]
)

In [ ]:
# 3. Fetch all embeddings from the "texts" collection
all_texts = collection_texts.get(include=["embeddings", "documents"])

# 4. Insert the projected text embeddings into the new "texts_images" collection
collection_texts_images.add(
    ids=all_texts["ids"],
    embeddings=all_texts["embeddings"],
    documents=all_texts["documents"],
    metadatas=[{"type": "text"} for _ in range(len(all_texts["ids"]))]
)

In [ ]:
# Just in case our device has gpu
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
model, _, _ = open_clip.create_model_and_transforms("hf-hub:laion/CLIP-ViT-L-14-laion2B-s32B-b82K")
tokenizer = open_clip.get_tokenizer("hf-hub:laion/CLIP-ViT-L-14-laion2B-s32B-b82K") # Tokenizer for texts
model.to(device)

In [ ]:
# Similarity search by the textual query "NieR:Automata"
query_text = "Games similar to NieR:Automata" # NieR: Automata does not exist in the current subset
q_vec = embed_text(model, tokenizer, query_text)

res_texts = collection_texts_images.query(
    query_embeddings=[q_vec],
    n_results=1,
    where={"type": "text"}, # Filter by metadata type
    include=["documents", "distances"]
)

res_images = collection_texts_images.query(
    query_embeddings=[q_vec],
    n_results=10,
    where={"type": "image"}, # Filter by metadata type
    include=["documents", "distances"]
)

In [ ]:
# Diplay top 1 similar text
get_text("trusted-zone", res_texts['documents'][0][0].split("/", 1)[1])

In [ ]:
# Diplay top 10 similar images
for i in range(10):
    display(get_image("trusted-zone", res_images['documents'][0][i].split("/", 1)[1]))
# The cover we want to see actually appears at the top 1 similar image